# Phase 2 clickbait Spoiling Improvement Pipeline — Step 1: Environment Setup

This cell clones the repository from the active development branch and switches to the directory.


In [ ]:
# Clone the repository and move strictly to the Version 2 active branch
!git clone https://github.com/aryonmt/clickbait-spoiling-IR-final-project.git
%cd clickbait-spoiling-IR-final-project
!git checkout phase2-multi-improvement

# Step 2: Install Pinned Dependencies

We install the stable requirements to maintain environment parity with the local LAPTOP.


In [ ]:
!pip install -r requirements.txt

# Step 3: Kaggle Inputs Directory Symlinking

Establishes symbolic links from Kaggle input datasets directly into the project's data directory.


In [ ]:
import os
import glob

# Ensure data directory is present in the cloned project root
os.makedirs("data", exist_ok=True)

# Locate dataset files in the Kaggle environment
train_paths = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
val_paths = glob.glob("/kaggle/input/**/validation.jsonl", recursive=True)

if train_paths:
    dest_train = "data/train.jsonl"
    if os.path.exists(dest_train) or os.path.islink(dest_train):
        os.remove(dest_train)
    os.symlink(train_paths[0], dest_train)
    print(f"Successfully linked train dataset to: {dest_train}")
else:
    print("[WARNING] 'train.jsonl' dataset was not found in Kaggle inputs.")

if val_paths:
    dest_val = "data/validation.jsonl"
    if os.path.exists(dest_val) or os.path.islink(dest_val):
        os.remove(dest_val)
    os.symlink(val_paths[0], dest_val)
    print(f"Successfully linked validation dataset to: {dest_val}")
else:
    print("[WARNING] 'validation.jsonl' dataset was not found in Kaggle inputs.")

# Step 4: Fine-tune Task 1 Classifier (RoBERTa-base)

Trains the 3-class spoiler classifier from scratch on Kaggle GPUs.


In [ ]:
!python -m main_transformer_task1 \
    --train_path data/train.jsonl \
    --val_path data/validation.jsonl \
    --model_name roberta-base \
    --output_dir results_task1_roberta \
    --epochs 5 \
    --lr 2e-5

# Step 5: Fine-tune Task 2 extractive QA utilizing Token-F1 (V2 Pipeline)

Trains the extractive question-answering transformer, tracking and saving based on token-level F1 scores.


In [ ]:
!python -m main_transformer_task2_qa \
    --train_path data/train.jsonl \
    --val_path data/validation.jsonl \
    --model_name deepset/roberta-base-squad2 \
    --output_dir results_qa \
    --epochs 3 \
    --lr 3e-5

# Step 6: Integrated Pipeline Inference with Lexical Retrieval for Multi-Type

Combines the class-weight tuned classifier and the newly optimized QA model with the Jaccard sentence retrieval system.


In [ ]:
import os
import glob

# Detect the best checkpoint folder for Task 1 RoBERTa automatically
task1_checkpoints = glob.glob("results_task1_roberta/checkpoint-*")
if task1_checkpoints:
    # Sort checkpoints by step number to find the latest/best one
    task1_checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    best_task1 = task1_checkpoints[-1]
    print(f"Detected best Task 1 checkpoint: {best_task1}")
else:
    best_task1 = "results_task1_roberta"
    print(f"[WARNING] No checkpoint folders found, defaulting to: {best_task1}")

# Run the integrated evaluation pipeline
cmd = f"python -m main_integrated_pipeline --val_path data/validation.jsonl --t1_checkpoint {best_task1} --t2_checkpoint results_qa --output_path run_integrated_pipeline.jsonl"
os.system(cmd)

# Step 7: Build Complete Phase 4 Visual Report Dashboard

Generates classification error distributions, correct/incorrect confidence scores, and packages outputs into a zip file.


In [ ]:
# Run report generation utilizing confidence metrics saved in the pipeline
!python -m main_generate_report \
    --predictions_path run_integrated_pipeline.jsonl \
    --val_path data/validation.jsonl

# Archive the complete reports folder into a zip file
!zip -r reports.zip reports/